# Seed Batch Scan Runner

Читает grouped target-table, проходит по всем `query_name`, матчится ко всем Steam item names, считает базу по первым 10 ask и сохраняет listings только по целевым seeds.

In [1]:
from pathlib import Path
import importlib.util

RUN_DIR = Path.cwd()
MODULE_PATH = RUN_DIR / "seed_batch_scan.py"
spec = importlib.util.spec_from_file_location("seed_batch_scan_runner_mod", MODULE_PATH)
seed_batch_scan = importlib.util.module_from_spec(spec)
spec.loader.exec_module(seed_batch_scan)

In [2]:
RUNTIME_JSON = RUN_DIR / "seed_batch_scan_runtime.json"
cfg = seed_batch_scan._load_runtime()

TARGET_CSV = seed_batch_scan._resolve_path(cfg.get("target_csv", "../data/seed_target_table.csv"), RUN_DIR)
ITEMS_PY = seed_batch_scan._resolve_path(cfg.get("items_py", "../../lists/screening_super_full.py"), RUN_DIR)
OUT_CSV = seed_batch_scan._resolve_path(cfg.get("out_csv", "data/seed_batch_scan_matches.csv"), RUN_DIR)
WRITE_MODE = str(cfg.get("write_mode", "create")).strip().lower()
SKIP_COMPLETED_QUERIES = seed_batch_scan._rt_bool(cfg, "skip_completed_queries", True)
QUERY_SUBSTRING = str(cfg.get("query_substring", "")).strip()
MAX_QUERIES = cfg.get("max_queries")
LIMIT = cfg.get("limit")
MAX_LISTINGS = cfg.get("max_listings")
steam_runtime_raw = str(cfg.get("steam_runtime", "")).strip()
STEAM_RUNTIME = seed_batch_scan._resolve_path(steam_runtime_raw, RUN_DIR) if steam_runtime_raw else seed_batch_scan.build_embedded_steam_runtime(cfg, RUN_DIR)

print(RUNTIME_JSON)
print({
    "target_csv": str(TARGET_CSV),
    "items_py": str(ITEMS_PY),
    "out_csv": str(OUT_CSV),
    "write_mode": WRITE_MODE,
    "skip_completed_queries": SKIP_COMPLETED_QUERIES,
    "query_substring": QUERY_SUBSTRING,
    "max_queries": MAX_QUERIES,
    "limit": LIMIT,
    "max_listings": MAX_LISTINGS,
    "steam_runtime": str(STEAM_RUNTIME),
})

c:\Roman\skins_roundtrip_v1\steam_seed_scan\seed_batch_scan\seed_batch_scan_runtime.json
{'target_csv': 'C:\\Roman\\skins_roundtrip_v1\\steam_seed_scan\\data\\seed_target_table.csv', 'items_py': 'C:\\Roman\\skins_roundtrip_v1\\lists\\screening_super_full.py', 'out_csv': 'C:\\Roman\\skins_roundtrip_v1\\steam_seed_scan\\seed_batch_scan\\data\\seed_batch_scan_matches.csv', 'write_mode': 'create', 'skip_completed_queries': True, 'query_substring': '', 'max_queries': None, 'limit': 100, 'max_listings': 200, 'steam_runtime': 'c:\\Roman\\skins_roundtrip_v1\\steam_seed_scan\\seed_batch_scan\\_embedded_steam_runtime.json'}


In [3]:
df, meta = seed_batch_scan.run_seed_batch_scan(
    target_csv=TARGET_CSV,
    items_py=ITEMS_PY,
    out_csv=OUT_CSV,
    limit=LIMIT,
    max_listings=MAX_LISTINGS,
    steam_runtime_path=STEAM_RUNTIME,
    query_substring=QUERY_SUBSTRING or None,
    max_queries=MAX_QUERIES,
    write_mode=WRITE_MODE,
    skip_completed_queries=SKIP_COMPLETED_QUERIES,
)
meta

  [steam_scm] "1/74 AK-47 | Case Hardened :: 1/10 AK-47 | Case Hardened (Battle-Scarred)": render start=0 count=100 (got 0/200)
  [seed_batch_scan] pause between items 4.8s (delay_between_skins_*)
  [steam_scm] "1/74 AK-47 | Case Hardened :: 2/10 AK-47 | Case Hardened (Factory New)": render start=0 count=100 (got 0/200)
  [seed_batch_scan] pause between items 4.2s (delay_between_skins_*)
  [steam_scm] "1/74 AK-47 | Case Hardened :: 3/10 AK-47 | Case Hardened (Field-Tested)": render start=0 count=100 (got 0/200)
  [seed_batch_scan] pause between items 4.9s (delay_between_skins_*)
  [steam_scm] "1/74 AK-47 | Case Hardened :: 4/10 AK-47 | Case Hardened (Minimal Wear)": render start=0 count=100 (got 0/200)
  [seed_batch_scan] pause between items 4.4s (delay_between_skins_*)
  [steam_scm] "1/74 AK-47 | Case Hardened :: 5/10 AK-47 | Case Hardened (Well-Worn)": render start=0 count=100 (got 0/200)
  [seed_batch_scan] pause between items 2.1s (delay_between_skins_*)
  [steam_scm] "1/74 AK-47 |

{'targets_total': 74,
 'queries_skipped': 0,
 'matched_item_total': 287,
 'unique_items_fetched': 287,
 'rows_saved': 153,
 'rows_saved_new': 153,
 'errors': 36,
 'out_csv': 'C:\\Roman\\skins_roundtrip_v1\\steam_seed_scan\\seed_batch_scan\\data\\seed_batch_scan_matches.csv',
 'target_csv': 'C:\\Roman\\skins_roundtrip_v1\\steam_seed_scan\\data\\seed_target_table.csv',
 'progress_csv': 'C:\\Roman\\skins_roundtrip_v1\\steam_seed_scan\\seed_batch_scan\\data\\seed_batch_scan_matches_progress.csv'}

In [4]:
df.head(50)

,query_name,market_hash_name,pattern_families,paint_seed,seed_confidence,overall_confidence,ask,ask_seller_net,base_ask_mean_10,base_ask_n,...,edge_ratio,float_value,listing_id,asset_id,scm_total_listings,converted_price,converted_fee,converted_currencyid,asset_properties_json,target_notes
0,MAC-10 | Case Hardened,MAC-10 | Case Hardened (Well-Worn),Blue Gem Top,890,0.92,0.92,348.04,302.65,281.133000,10,...,1.827162,0.444418,806844236162352093,49730926511,10,30265,4539,2003,NaN,Из дешёвых CH-носителей один из лучших кандида...
1,MAC-10 | Case Hardened,MAC-10 | Case Hardened (Minimal Wear),Blue Gem Top,95,0.92,0.92,423.78,368.51,224.245000,10,...,1.116617,0.140939,657102640620647770,48985966416,10,36851,5527,2003,NaN,Из дешёвых CH-носителей один из лучших кандида...
2,Glock-18 | Gamma Doppler,Glock-18 | Gamma Doppler (Minimal Wear),High Tier,534,0.80,0.80,243.69,211.91,154.400000,10,...,1.217572,0.146658,630079774426595649,48344658108,91,21191,3178,2003,NaN,"Не нож, но дорогой и вполне pattern-sensitive ..."
3,Glock-18 | Gamma Doppler,Glock-18 | Gamma Doppler (Field-Tested),High Tier,362,0.80,0.80,117.03,101.78,106.803000,10,...,1.737836,0.310249,545639912146526138,50274613639,77,10178,1525,2003,NaN,"Не нож, но дорогой и вполне pattern-sensitive ..."
4,MAC-10 | Case Hardened,MAC-10 | Case Hardened (Battle-Scarred),Blue Gem Top,239,0.92,0.92,287.50,250.00,168.105000,6,...,1.046496,0.576493,507359315990741839,50721740187,6,25000,3750,2003,NaN,Из дешёвых CH-носителей один из лучших кандида...
5,MAC-10 | Case Hardened,MAC-10 | Case Hardened (Field-Tested),Blue Gem Top,199,0.92,0.92,150.55,130.92,121.095000,10,...,1.413052,0.338443,639089510337060006,47514538920,20,13092,1963,2003,NaN,Из дешёвых CH-носителей один из лучших кандида...
6,MAC-10 | Case Hardened,MAC-10 | Case Hardened (Field-Tested),Blue Gem Top,199,0.92,0.92,162.52,141.33,121.095000,10,...,1.235325,0.154415,626701440920785116,41082472862,20,14133,2119,2003,NaN,Из дешёвых CH-носителей один из лучших кандида...
7,MAC-10 | Case Hardened,MAC-10 | Case Hardened (Field-Tested),Blue Gem Top,251,0.92,0.92,286.69,249.30,121.095000,10,...,0.689560,0.294410,635709273938848447,48300625808,20,24930,3739,2003,NaN,Из дешёвых CH-носителей один из лучших кандида...
8,Glock-18 | Gamma Doppler,Glock-18 | Gamma Doppler (Field-Tested),High Tier,408,0.80,0.80,172.50,150.00,106.803000,10,...,0.857443,0.199932,652598587799781408,43254280329,77,15000,2250,2003,NaN,"Не нож, но дорогой и вполне pattern-sensitive ..."
9,Glock-18 | Gamma Doppler,Glock-18 | Gamma Doppler (Battle-Scarred),High Tier,452,0.80,0.80,373.93,325.17,165.832000,10,...,0.552194,0.472512,756172323970953761,48168557871,10,32517,4876,2003,NaN,"Не нож, но дорогой и вполне pattern-sensitive ..."
